In [2]:
from osgeo import gdal
import wradlib as wrl

file_path = "D:\\Data\\BHP190610110229.RAWSU69"
fcontent = wrl.io.read_iris(file_path)
fcontent = fcontent['data']

In [3]:
import math
%matplotlib widget
import matplotlib.pyplot as plt

def convert_radian(angle):
    angle*= math.pi/180
    return angle
def plot(df):
    fig = plt.figure()
    ax = plt.axes(projection='3d')
    ax.scatter3D(df['X'], df['Y'], df['Z'], marker = 1)
    fig.show()
    

In [5]:
import pandas as pd
import numpy as np
import numpy.ma as ma

cart = []

for key1 in fcontent:#10 readings at different elevations
    sweep_reading = fcontent[key1]['sweep_data']
    db_vel = sweep_reading['DB_VEL']#extract only dbvel data from reading
    db_dbz = sweep_reading['DB_DBZ']#extract only dbz data from reading
    db_width = sweep_reading['DB_WIDTH']
    db_dbz['azi_start'][0]-=360
    dbz = pd.DataFrame.from_dict({'azi_start':db_dbz['azi_start'],'azi_stop':db_dbz['azi_stop'],'ele_start':db_dbz['ele_start'],'ele_stop':db_dbz['ele_stop']})#convert dbz data to dataframe
    dbz['azi_mean'] = dbz[['azi_start','azi_stop']].mean(axis=1)#compute mean of azi
    dbz['ele_mean'] = dbz[['ele_start','ele_stop']].mean(axis=1)#compute mean of ele
    dbz['azi_mean'] = dbz['azi_mean'].apply(convert_radian)#convert mean from degree to radian
    dbz['ele_mean'] = dbz['ele_mean'].apply(convert_radian)#convert mean from degree to radian
    dbz_data = db_dbz['data']#extract data list (2-d) (360x500)
    dbvel_data = db_vel['data']#extract data list (2-d) (360x500)
    dbwidth_data = db_width['data']
    x,y,z,dbz_v,dbvel_v,dbwidth_v = [],[],[],[],[],[]#declare lists for coordinates and values
    for j in range(len(dbz_data)):
        azi = dbz['azi_mean'][j]#read azi for particular ray
        ele = dbz['ele_mean'][j]#read ele for particular ray
        dbzvalue = dbz_data[j]
        dbvelvalue = dbvel_data[j]
        dbwidthvalue = dbwidth_data[j]
        for k in range(len(dbzvalue)):#process all (500) reading of particular ray
            i = k*0.5
            Z = i*math.sin(ele)
            Y = i*math.cos(ele)*math.cos(azi)
            X = i*math.cos(ele)*math.sin(azi)
            x.append(X)
            y.append(Y)
            z.append(Z)
            dbz_v.append(dbzvalue[k])
            if dbvelvalue[k] is ma.masked:
                dbvel_v.append(0)
            else:
                dbvel_v.append(float(dbvelvalue[k]))
            dbwidth_v.append(dbwidthvalue[k])
    cart.append(pd.DataFrame({'X':x,'Y':y,'Z':z,'DBZ':dbz_v,'DBVEL':dbvel_v,'DBWIDTH':dbwidth_v}))


l = pd.concat(cart)
l.to_csv('l.csv')